# 📈 Horizon Books — Forecasting Exploration Notebook

This notebook runs **locally** against the sample CSV data and applies the same
**Holt-Winters exponential smoothing** models used by `notebooks/04_Forecasting.py`
in the Fabric pipeline.

## Forecast Models

| # | Model | Table | Grain | Horizon |
|---|-------|-------|-------|---------|
| 1 | Sales Revenue | `ForecastSalesRevenue` | Channel | 6 months |
| 2 | Genre Demand | `ForecastGenreDemand` | Genre | 6 months |
| 3 | Financial P&L | `ForecastFinancial` | Account Type | 6 months |
| 4 | Inventory Demand | `ForecastInventoryDemand` | Warehouse | 6 months |
| 5 | Workforce Planning | `ForecastWorkforce` | Metric | 6 months |

All models use **statsmodels `ExponentialSmoothing`** with automatic fallback:
- Full Holt-Winters (additive/multiplicative seasonality) if ≥24 data points
- Holt linear trend if 12–23 data points
- Weighted moving average + linear trend if 3–11 data points
- Naïve last-value if <3 data points

In [ ]:
# Cell 1: Setup — imports, configuration, helpers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from datetime import datetime
from pathlib import Path
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

# ── Configuration (matches Fabric notebook 04_Forecasting.py) ──
FORECAST_HORIZON = 6           # months ahead
CONFIDENCE_LEVEL = 0.95        # 95% confidence interval
MIN_HISTORY_MONTHS = 12        # minimum data points for Holt-Winters
SEASONAL_PERIODS = 12          # monthly seasonality

# ── Paths ──
ROOT = Path("..")  # repo root relative to Forecasting/
DATA = ROOT / "SampleData"

# ── Color palette (matches Power BI report theme) ──
COLORS = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B",
          "#44BBA4", "#E94F37", "#393E41", "#8D6A9F", "#5FAD56"]

print(f"Forecast horizon : {FORECAST_HORIZON} months")
print(f"Confidence level : {CONFIDENCE_LEVEL*100:.0f}%")
print(f"Min history      : {MIN_HISTORY_MONTHS} months")
print(f"Data path        : {DATA.resolve()}")

In [ ]:
# Cell 2: Load sample data
df_orders    = pd.read_csv(DATA / "Operations" / "FactOrders.csv", parse_dates=["OrderDate"])
df_books     = pd.read_csv(DATA / "Operations" / "DimBooks.csv")
df_inventory = pd.read_csv(DATA / "Operations" / "FactInventory.csv", parse_dates=["SnapshotDate"])
df_finance   = pd.read_csv(DATA / "Finance" / "FactFinancialTransactions.csv", parse_dates=["TransactionDate"])
df_accounts  = pd.read_csv(DATA / "Finance" / "DimAccounts.csv")
df_payroll   = pd.read_csv(DATA / "HR" / "FactPayroll.csv", parse_dates=["PayDate"])
df_recruit   = pd.read_csv(DATA / "HR" / "FactRecruitment.csv", parse_dates=["OpenDate"])

# Enrich orders with Genre from DimBooks
df_orders = df_orders.merge(df_books[["BookID", "Genre"]], on="BookID", how="left")

# Enrich finance with AccountType from DimAccounts
df_finance = df_finance.merge(df_accounts[["AccountID", "AccountType"]], on="AccountID", how="left")

print(f"Orders:       {len(df_orders):>6,} rows  ({df_orders['OrderDate'].min().date()} – {df_orders['OrderDate'].max().date()})")
print(f"Books:        {len(df_books):>6,} rows")
print(f"Inventory:    {len(df_inventory):>6,} rows")
print(f"Finance:      {len(df_finance):>6,} rows  ({df_finance['TransactionDate'].min().date()} – {df_finance['TransactionDate'].max().date()})")
print(f"Payroll:      {len(df_payroll):>6,} rows  ({df_payroll['PayDate'].min().date()} – {df_payroll['PayDate'].max().date()})")
print(f"Recruitment:  {len(df_recruit):>6,} rows")

In [ ]:
# Cell 3: Forecasting engine (mirrors 04_Forecasting.py logic)

def holt_winters_forecast(ts_series, horizon, seasonal_periods=SEASONAL_PERIODS):
    """
    Apply Holt-Winters Exponential Smoothing to a pandas Series.
    Falls back through progressively simpler models if data is insufficient.
    Returns (forecast_values, lower_bound, upper_bound, model_name).
    """
    values = ts_series.dropna().values.astype(float)
    n = len(values)
    z = 1.96  # 95% CI

    # Attempt 1: Full Holt-Winters with seasonal component
    if n >= seasonal_periods * 2:
        for seasonal in ["add", "mul"]:
            for trend in ["add", "mul"]:
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        model = ExponentialSmoothing(
                            values, trend=trend, seasonal=seasonal,
                            seasonal_periods=seasonal_periods,
                            initialization_method="estimated"
                        ).fit(optimized=True, use_brute=True, maxiter=500)
                    fcast = model.forecast(horizon)
                    residuals = values - model.fittedvalues
                    sigma = np.std(residuals)
                    lower = fcast - z * sigma * np.sqrt(np.arange(1, horizon + 1))
                    upper = fcast + z * sigma * np.sqrt(np.arange(1, horizon + 1))
                    return fcast, lower, upper, f"HoltWinters_{trend}_{seasonal}"
                except Exception:
                    continue

    # Attempt 2: Additive trend only (no seasonality)
    if n >= 3:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = ExponentialSmoothing(
                    values, trend="add", seasonal=None
                ).fit(optimized=True, use_brute=True, maxiter=500)
            fcast = model.forecast(horizon)
            residuals = values - model.fittedvalues
            sigma = np.std(residuals)
            lower = fcast - z * sigma * np.sqrt(np.arange(1, horizon + 1))
            upper = fcast + z * sigma * np.sqrt(np.arange(1, horizon + 1))
            return fcast, lower, upper, "HoltLinearTrend"
        except Exception:
            pass

    # Attempt 3: Weighted moving average + linear trend
    if n >= 3:
        weights = np.arange(1, n + 1, dtype=float)
        weights /= weights.sum()
        wma = np.average(values, weights=weights)
        slope = np.polyfit(np.arange(n), values, 1)[0]
        fcast = np.array([wma + slope * (i + 1) for i in range(horizon)])
        sigma = np.std(values[-min(n, 6):])
        lower = fcast - z * sigma
        upper = fcast + z * sigma
        return fcast, lower, upper, "WMA_LinearTrend"

    # Fallback: Na\u00efve
    last_val = values[-1] if n > 0 else 0.0
    fcast = np.full(horizon, last_val)
    return fcast, fcast * 0.8, fcast * 1.2, "Naive"


def compute_mape(actual, predicted):
    """Mean Absolute Percentage Error (safe for zeros)."""
    mask = actual != 0
    if mask.sum() == 0:
        return None
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)


def backtest_mape(ts, forecast_fn, holdout=3):
    """Back-test MAPE on the last `holdout` months."""
    if len(ts) < holdout * 2:
        return None
    try:
        bt_fcast, _, _, _ = forecast_fn(ts.iloc[:-holdout], holdout)
        return compute_mape(ts.iloc[-holdout:].values, bt_fcast)
    except Exception:
        return None


def plot_forecast(ax, dates_actual, values_actual, dates_forecast, values_forecast,
                  lower, upper, title, ylabel, color, model_name, mape=None):
    """Plot a single forecast series with confidence band."""
    ax.plot(dates_actual, values_actual, color=color, linewidth=2, label="Actual")
    ax.plot(dates_forecast, values_forecast, color=color, linewidth=2,
            linestyle="--", marker="o", markersize=4, label="Forecast")
    ax.fill_between(dates_forecast, lower, upper, alpha=0.15, color=color, label="95% CI")
    # Connect actual to forecast
    ax.plot([dates_actual.iloc[-1], dates_forecast[0]],
            [values_actual.iloc[-1], values_forecast[0]],
            color=color, linewidth=1.5, linestyle="--", alpha=0.5)
    mape_str = f" | MAPE={mape:.1f}%" if mape else ""
    ax.set_title(f"{title}\n({model_name}{mape_str})", fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.legend(loc="upper left", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.tick_params(axis="x", rotation=45)

print("✓ Forecasting engine loaded")

---
## 1. Sales Revenue Forecast by Channel

Predicts monthly revenue per sales channel (Online, Retail, Wholesale, etc.)
using Holt-Winters exponential smoothing with a 6-month horizon.

In [ ]:
# Cell 4: Sales Revenue Forecast by Channel

df_orders["OrderMonth"] = df_orders["OrderDate"].dt.to_period("M").dt.to_timestamp()

monthly_channel = (
    df_orders.groupby(["OrderMonth", "Channel"])
    .agg(Revenue=("TotalAmount", "sum"),
         Orders=("OrderID", "count"),
         Customers=("CustomerID", "nunique"))
    .reset_index()
    .sort_values(["Channel", "OrderMonth"])
)

channels = monthly_channel["Channel"].unique()
last_date = monthly_channel["OrderMonth"].max()

n_channels = len(channels)
fig, axes = plt.subplots(n_channels, 1, figsize=(14, 4 * n_channels), squeeze=False)
fig.suptitle("Sales Revenue Forecast by Channel", fontsize=16, fontweight="bold", y=1.01)

sales_results = []
sales_summary = []

for idx, ch in enumerate(sorted(channels)):
    ch_data = monthly_channel[monthly_channel["Channel"] == ch].sort_values("OrderMonth")
    ts = ch_data.set_index("OrderMonth")["Revenue"]

    if len(ts) < 3:
        print(f"  \u26a0 {ch}: only {len(ts)} months \u2014 skipping")
        continue

    fcast, lower, upper, model_name = holt_winters_forecast(ts, FORECAST_HORIZON)
    mape = backtest_mape(ts, holt_winters_forecast)

    # Forecast dates
    fc_dates = pd.date_range(start=last_date + pd.DateOffset(months=1),
                             periods=FORECAST_HORIZON, freq="MS")

    plot_forecast(axes[idx, 0], ch_data["OrderMonth"], ch_data["Revenue"],
                  fc_dates, fcast, lower, upper,
                  f"{ch}", "Revenue ($)", COLORS[idx % len(COLORS)],
                  model_name, mape)

    sales_summary.append({
        "Channel": ch, "Model": model_name,
        "Months": len(ts), "MAPE_3m": f"{mape:.1f}%" if mape else "N/A",
        "Next Month Forecast": f"${fcast[0]:,.0f}",
        "6-Month Total": f"${sum(fcast):,.0f}"
    })

plt.tight_layout()
plt.show()

pd.DataFrame(sales_summary)

---
## 2. Genre Demand Forecast

Predicts monthly unit demand per book genre. Uses the `Genre` column
from `DimBooks` joined to orders.

In [ ]:
# Cell 5: Genre Demand Forecast

monthly_genre = (
    df_orders[df_orders["Genre"].notna()]
    .groupby(["OrderMonth", "Genre"])
    .agg(UnitDemand=("Quantity", "sum"),
         Revenue=("TotalAmount", "sum"))
    .reset_index()
    .sort_values(["Genre", "OrderMonth"])
)

genres = sorted(monthly_genre["Genre"].unique())
last_date_g = monthly_genre["OrderMonth"].max()

n_genres = len(genres)
ncols = 2
nrows = (n_genres + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
fig.suptitle("Genre Demand Forecast (Units)", fontsize=16, fontweight="bold", y=1.01)
axes_flat = axes.flatten()

genre_summary = []

for idx, genre in enumerate(genres):
    g_data = monthly_genre[monthly_genre["Genre"] == genre].sort_values("OrderMonth")
    ts = g_data.set_index("OrderMonth")["UnitDemand"]

    if len(ts) < 3:
        continue

    fcast, lower, upper, model_name = holt_winters_forecast(ts, FORECAST_HORIZON)
    mape = backtest_mape(ts, holt_winters_forecast)

    fc_dates = pd.date_range(start=last_date_g + pd.DateOffset(months=1),
                             periods=FORECAST_HORIZON, freq="MS")

    plot_forecast(axes_flat[idx], g_data["OrderMonth"], g_data["UnitDemand"],
                  fc_dates, fcast, lower, upper,
                  genre, "Units", COLORS[idx % len(COLORS)],
                  model_name, mape)

    genre_summary.append({
        "Genre": genre, "Model": model_name,
        "Months": len(ts), "MAPE_3m": f"{mape:.1f}%" if mape else "N/A",
        "Next Month": f"{fcast[0]:,.0f} units",
        "6-Month Total": f"{sum(np.maximum(fcast, 0)):,.0f} units"
    })

# Hide unused subplots
for j in range(idx + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.show()

pd.DataFrame(genre_summary)

---
## 3. Financial P&L Forecast

Forecasts monthly P&L amounts by `AccountType` (Revenue, COGS, Operating Expenses, etc.).
Expense categories (negative amounts) are modeled in absolute terms
then sign-restored.

In [ ]:
# Cell 6: Financial P&L Forecast

df_finance["TxnMonth"] = df_finance["TransactionDate"].dt.to_period("M").dt.to_timestamp()

monthly_pl = (
    df_finance[df_finance["AccountType"].notna()]
    .groupby(["TxnMonth", "AccountType"])
    .agg(Amount=("Amount", "sum"),
         TransactionCount=("TransactionID", "count"))
    .reset_index()
    .sort_values(["AccountType", "TxnMonth"])
)

categories = sorted(monthly_pl["AccountType"].unique())
last_date_f = monthly_pl["TxnMonth"].max()

n_cats = len(categories)
ncols = 2
nrows = (n_cats + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
fig.suptitle("Financial P&L Forecast by Account Type", fontsize=16, fontweight="bold", y=1.01)
axes_flat = axes.flatten() if n_cats > 1 else [axes]

fin_summary = []

for idx, cat in enumerate(categories):
    c_data = monthly_pl[monthly_pl["AccountType"] == cat].sort_values("TxnMonth")
    ts = c_data.set_index("TxnMonth")["Amount"]

    if len(ts) < 3:
        continue

    # Model absolute values for expense categories, restore sign after
    sign = 1 if ts.mean() >= 0 else -1
    ts_model = ts.abs() if sign < 0 else ts

    fcast, lower, upper, model_name = holt_winters_forecast(ts_model, FORECAST_HORIZON)

    if sign < 0:
        fcast = -fcast
        lower_orig = lower.copy()
        lower = -upper
        upper = -lower_orig

    fc_dates = pd.date_range(start=last_date_f + pd.DateOffset(months=1),
                             periods=FORECAST_HORIZON, freq="MS")

    plot_forecast(axes_flat[idx], c_data["TxnMonth"], c_data["Amount"],
                  fc_dates, fcast, lower, upper,
                  cat, "Amount ($)", COLORS[idx % len(COLORS)],
                  model_name)

    fin_summary.append({
        "Account Type": cat, "Model": model_name,
        "Months": len(ts),
        "Next Month": f"${fcast[0]:,.0f}",
        "6-Month Total": f"${sum(fcast):,.0f}"
    })

for j in range(idx + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.show()

pd.DataFrame(fin_summary)

---
## 4. Inventory Demand Forecast by Warehouse

Forecasts future unit demand per warehouse and compares against current
inventory levels to identify potential stockout risks.

In [ ]:
# Cell 7: Inventory Demand Forecast by Warehouse

monthly_wh = (
    df_orders[df_orders["WarehouseID"].notna()]
    .groupby(["OrderMonth", "WarehouseID"])
    .agg(UnitsDemanded=("Quantity", "sum"),
         Revenue=("TotalAmount", "sum"),
         OrderCount=("OrderID", "count"))
    .reset_index()
    .sort_values(["WarehouseID", "OrderMonth"])
)

# Latest inventory snapshot per warehouse
latest_inv = (
    df_inventory.sort_values("SnapshotDate")
    .groupby("WarehouseID")
    .agg(CurrentStock=("QuantityOnHand", "sum"))
    .reset_index()
)
stock_map = dict(zip(latest_inv["WarehouseID"], latest_inv["CurrentStock"]))

warehouses = sorted(monthly_wh["WarehouseID"].unique())
last_date_w = monthly_wh["OrderMonth"].max()

n_wh = len(warehouses)
fig, axes = plt.subplots(n_wh, 1, figsize=(14, 4 * n_wh), squeeze=False)
fig.suptitle("Inventory Demand Forecast by Warehouse", fontsize=16, fontweight="bold", y=1.01)

inv_summary = []

for idx, wh in enumerate(warehouses):
    wh_data = monthly_wh[monthly_wh["WarehouseID"] == wh].sort_values("OrderMonth")
    ts = wh_data.set_index("OrderMonth")["UnitsDemanded"]

    if len(ts) < 3:
        continue

    fcast, lower, upper, model_name = holt_winters_forecast(ts, FORECAST_HORIZON)
    mape = backtest_mape(ts, holt_winters_forecast)
    current_stock = stock_map.get(wh, 0)
    cumulative_demand = sum(np.maximum(fcast, 0))
    avg_monthly = cumulative_demand / FORECAST_HORIZON if cumulative_demand > 0 else 1
    stock_cover_months = current_stock / avg_monthly if avg_monthly > 0 else 99

    fc_dates = pd.date_range(start=last_date_w + pd.DateOffset(months=1),
                             periods=FORECAST_HORIZON, freq="MS")

    ax = axes[idx, 0]
    plot_forecast(ax, wh_data["OrderMonth"], wh_data["UnitsDemanded"],
                  fc_dates, fcast, lower, upper,
                  f"{wh}", "Units Demanded", COLORS[idx % len(COLORS)],
                  model_name, mape)

    # Add stock coverage annotation
    risk = "\u26a0 LOW" if stock_cover_months < 3 else "\u2713 OK"
    ax.annotate(f"Stock: {current_stock:,.0f} | Cover: {stock_cover_months:.1f}mo {risk}",
                xy=(0.98, 0.95), xycoords="axes fraction", ha="right", va="top",
                fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="#FFCCCC" if stock_cover_months < 3 else "#CCFFCC",
                          alpha=0.8))

    inv_summary.append({
        "Warehouse": wh, "Model": model_name,
        "Current Stock": f"{current_stock:,.0f}",
        "6-Mo Demand": f"{cumulative_demand:,.0f}",
        "Stock Cover": f"{stock_cover_months:.1f} months",
        "Risk": "\u26a0" if stock_cover_months < 3 else "\u2713"
    })

plt.tight_layout()
plt.show()

pd.DataFrame(inv_summary)

---
## 5. Workforce Planning Forecast

Two sub-models:
- **Hiring pipeline**: Forecasts monthly job openings and hires from recruitment data
- **Payroll costs**: Forecasts total payroll and headcount trends

In [ ]:
# Cell 8: Workforce Planning Forecast

# ── Hiring pipeline ──
df_recruit["HireMonth"] = df_recruit["OpenDate"].dt.to_period("M").dt.to_timestamp()
monthly_hiring = (
    df_recruit.groupby("HireMonth")
    .agg(Openings=("RequisitionID", "count"),
         Hires=("Status", lambda x: (x == "Filled").sum()))
    .reset_index()
    .sort_values("HireMonth")
)

# ── Payroll costs ──
monthly_payroll = (
    df_payroll.groupby(df_payroll["PayDate"].dt.to_period("M").dt.to_timestamp().rename("PayMonth"))
    .agg(TotalPayroll=("NetPay", "sum"),
         Headcount=("EmployeeID", "nunique"))
    .reset_index()
    .sort_values("PayMonth")
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Workforce Planning Forecast", fontsize=16, fontweight="bold", y=1.01)

wf_summary = []

# (a) Openings forecast
ts_open = monthly_hiring.set_index("HireMonth")["Openings"]
if len(ts_open) >= 3:
    fcast_o, lo_o, up_o, model_o = holt_winters_forecast(ts_open, FORECAST_HORIZON)
    fc_dates = pd.date_range(start=ts_open.index.max() + pd.DateOffset(months=1),
                             periods=FORECAST_HORIZON, freq="MS")
    plot_forecast(axes[0, 0], monthly_hiring["HireMonth"], monthly_hiring["Openings"],
                  fc_dates, fcast_o, lo_o, up_o,
                  "Job Openings", "Count", COLORS[0], model_o)
    wf_summary.append({"Metric": "Openings", "Model": model_o, "Months": len(ts_open),
                       "Next Month": f"{max(fcast_o[0], 0):.0f}"})

# (b) Hires forecast
ts_hires = monthly_hiring.set_index("HireMonth")["Hires"]
if len(ts_hires) >= 3:
    fcast_h, lo_h, up_h, model_h = holt_winters_forecast(ts_hires, FORECAST_HORIZON)
    fc_dates = pd.date_range(start=ts_hires.index.max() + pd.DateOffset(months=1),
                             periods=FORECAST_HORIZON, freq="MS")
    plot_forecast(axes[0, 1], monthly_hiring["HireMonth"], monthly_hiring["Hires"],
                  fc_dates, fcast_h, lo_h, up_h,
                  "Monthly Hires", "Count", COLORS[1], model_h)
    wf_summary.append({"Metric": "Hires", "Model": model_h, "Months": len(ts_hires),
                       "Next Month": f"{max(fcast_h[0], 0):.0f}"})

# (c) Payroll forecast
ts_pay = monthly_payroll.set_index("PayMonth")["TotalPayroll"]
if len(ts_pay) >= 3:
    fcast_p, lo_p, up_p, model_p = holt_winters_forecast(ts_pay, FORECAST_HORIZON)
    fc_dates = pd.date_range(start=ts_pay.index.max() + pd.DateOffset(months=1),
                             periods=FORECAST_HORIZON, freq="MS")
    plot_forecast(axes[1, 0], monthly_payroll["PayMonth"], monthly_payroll["TotalPayroll"],
                  fc_dates, fcast_p, lo_p, up_p,
                  "Total Payroll", "Amount ($)", COLORS[2], model_p)
    wf_summary.append({"Metric": "TotalPayroll", "Model": model_p, "Months": len(ts_pay),
                       "Next Month": f"${max(fcast_p[0], 0):,.0f}"})

# (d) Headcount forecast
ts_hc = monthly_payroll.set_index("PayMonth")["Headcount"]
if len(ts_hc) >= 3:
    fcast_hc, lo_hc, up_hc, model_hc = holt_winters_forecast(ts_hc, FORECAST_HORIZON)
    fc_dates = pd.date_range(start=ts_hc.index.max() + pd.DateOffset(months=1),
                             periods=FORECAST_HORIZON, freq="MS")
    plot_forecast(axes[1, 1], monthly_payroll["PayMonth"], monthly_payroll["Headcount"],
                  fc_dates, fcast_hc, lo_hc, up_hc,
                  "Headcount", "Employees", COLORS[3], model_hc)
    wf_summary.append({"Metric": "Headcount", "Model": model_hc, "Months": len(ts_hc),
                       "Next Month": f"{max(fcast_hc[0], 0):.0f}"})

plt.tight_layout()
plt.show()

pd.DataFrame(wf_summary)

---
## 6. Cross-Model Summary & Comparison

Aggregated view of all forecast models, their accuracy, and key projections.

In [ ]:
# Cell 9: Cross-model summary

# Collect all time series and their forecasts for a composite view
print("=" * 60)
print("  FORECAST MODEL SUMMARY")
print("=" * 60)

all_models = []

# Sales Revenue (aggregate)
ts_rev = monthly_channel.groupby("OrderMonth")["Revenue"].sum()
if len(ts_rev) >= 3:
    fcast_r, _, _, model_r = holt_winters_forecast(ts_rev, FORECAST_HORIZON)
    mape_r = backtest_mape(ts_rev, holt_winters_forecast)
    all_models.append({
        "Forecast": "Total Revenue", "Model": model_r,
        "History (months)": len(ts_rev),
        "MAPE": f"{mape_r:.1f}%" if mape_r else "N/A",
        "Next Month": f"${fcast_r[0]:,.0f}",
        "6-Month Projection": f"${sum(np.maximum(fcast_r, 0)):,.0f}"
    })

# Total Unit Demand
ts_units = monthly_genre.groupby("OrderMonth")["UnitDemand"].sum()
if len(ts_units) >= 3:
    fcast_u, _, _, model_u = holt_winters_forecast(ts_units, FORECAST_HORIZON)
    mape_u = backtest_mape(ts_units, holt_winters_forecast)
    all_models.append({
        "Forecast": "Total Unit Demand", "Model": model_u,
        "History (months)": len(ts_units),
        "MAPE": f"{mape_u:.1f}%" if mape_u else "N/A",
        "Next Month": f"{fcast_u[0]:,.0f} units",
        "6-Month Projection": f"{sum(np.maximum(fcast_u, 0)):,.0f} units"
    })

# Net Financial
ts_net = monthly_pl.groupby("TxnMonth")["Amount"].sum()
if len(ts_net) >= 3:
    fcast_n, _, _, model_n = holt_winters_forecast(ts_net, FORECAST_HORIZON)
    all_models.append({
        "Forecast": "Net Financial", "Model": model_n,
        "History (months)": len(ts_net),
        "MAPE": "N/A",
        "Next Month": f"${fcast_n[0]:,.0f}",
        "6-Month Projection": f"${sum(fcast_n):,.0f}"
    })

# Total Payroll
if len(ts_pay) >= 3:
    all_models.append({
        "Forecast": "Total Payroll", "Model": model_p,
        "History (months)": len(ts_pay),
        "MAPE": "N/A",
        "Next Month": f"${max(fcast_p[0], 0):,.0f}",
        "6-Month Projection": f"${sum(np.maximum(fcast_p, 0)):,.0f}"
    })

summary_df = pd.DataFrame(all_models)
print(summary_df.to_string(index=False))
print(f"\nForecast horizon: {FORECAST_HORIZON} months")
print(f"Generated at: {datetime.utcnow():%Y-%m-%d %H:%M:%S} UTC")

summary_df

In [ ]:
# Cell 10: Composite forecast visualization

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Horizon Books \u2014 6-Month Forecast Overview",
             fontsize=16, fontweight="bold", y=1.01)

# (a) Total Revenue
ts_rev_idx = ts_rev.reset_index()
ts_rev_idx.columns = ["Month", "Revenue"]
fc_dates = pd.date_range(start=ts_rev.index.max() + pd.DateOffset(months=1),
                         periods=FORECAST_HORIZON, freq="MS")
fcast_r, lower_r, upper_r, _ = holt_winters_forecast(ts_rev, FORECAST_HORIZON)
plot_forecast(axes[0, 0], ts_rev_idx["Month"], ts_rev_idx["Revenue"],
              fc_dates, fcast_r, lower_r, upper_r,
              "Total Revenue", "Revenue ($)", COLORS[0], model_r)

# (b) Total Unit Demand
ts_units_idx = ts_units.reset_index()
ts_units_idx.columns = ["Month", "Units"]
fc_dates_u = pd.date_range(start=ts_units.index.max() + pd.DateOffset(months=1),
                           periods=FORECAST_HORIZON, freq="MS")
fcast_u2, lower_u2, upper_u2, _ = holt_winters_forecast(ts_units, FORECAST_HORIZON)
plot_forecast(axes[0, 1], ts_units_idx["Month"], ts_units_idx["Units"],
              fc_dates_u, fcast_u2, lower_u2, upper_u2,
              "Total Unit Demand", "Units", COLORS[1], model_u)

# (c) Net Financial
ts_net_idx = ts_net.reset_index()
ts_net_idx.columns = ["Month", "Amount"]
fc_dates_n = pd.date_range(start=ts_net.index.max() + pd.DateOffset(months=1),
                           periods=FORECAST_HORIZON, freq="MS")
fcast_n2, lower_n2, upper_n2, _ = holt_winters_forecast(ts_net, FORECAST_HORIZON)
plot_forecast(axes[1, 0], ts_net_idx["Month"], ts_net_idx["Amount"],
              fc_dates_n, fcast_n2, lower_n2, upper_n2,
              "Net Financial Position", "Amount ($)", COLORS[2], model_n)

# (d) Payroll
plot_forecast(axes[1, 1], monthly_payroll["PayMonth"], monthly_payroll["TotalPayroll"],
              pd.date_range(start=ts_pay.index.max() + pd.DateOffset(months=1),
                            periods=FORECAST_HORIZON, freq="MS"),
              fcast_p, lo_p, up_p,
              "Total Payroll", "Amount ($)", COLORS[3], model_p)

plt.tight_layout()
plt.show()

print("\n=== Forecasting Exploration Complete ===")